# Journeys and Transitions

A **journey** in JuPedSim is a sequence of stages (waypoints, queues, exits) that
defines the route an agent follows.  A **transition** specifies which stage an agent
moves to when it completes the current one.  JuPedSim provides four transition
factories:

| Factory | Behaviour |
|---|---|
| `create_fixed_transition(stage_id)` | Always go to the same next stage. |
| `create_round_robin_transition([(stage_id, weight), ...])` | Cycle through choices by weight. |
| `create_least_targeted_transition([stage_id, ...])` | Go to whichever choice currently has the fewest agents heading towards it. |
| `create_none_transition()` | Agent is removed from the simulation when it completes this stage. |

Transitions are attached to a `JourneyDescription` before calling
`simulation.add_journey(journey)`.


## Example 1 – Fixed transitions through two waypoints

Agents walk from the left side of a corridor, pass through two waypoints in
sequence, and exit on the right.  Each waypoint uses a fixed transition to
advance to the next stage.


In [ ]:
import pathlib

import jupedsim as jps
import shapely

trajectory_file = pathlib.Path("journey.sqlite")

geometry = shapely.Polygon([(0, 0), (20, 0), (20, 10), (0, 10)])
simulation = jps.Simulation(
    model=jps.CollisionFreeSpeedModel(),
    geometry=geometry,
    trajectory_writer=jps.SqliteTrajectoryWriter(
        output_file=trajectory_file, commit_every_nth_write=1
    ),
)

wp1 = simulation.add_waypoint_stage((10, 8), 0.5)
wp2 = simulation.add_waypoint_stage((10, 2), 0.5)
exit_id = simulation.add_exit_stage([(19, 4), (20, 4), (20, 6), (19, 6)])

journey = jps.JourneyDescription([wp1, wp2, exit_id])
journey.set_transition_for_stage(
    wp1, jps.Transition.create_fixed_transition(wp2)
)
journey.set_transition_for_stage(
    wp2, jps.Transition.create_fixed_transition(exit_id)
)
journey_id = simulation.add_journey(journey)

positions = jps.distributions.distribute_by_number(
    polygon=shapely.Polygon([(0.5, 0.5), (3, 0.5), (3, 9.5), (0.5, 9.5)]),
    number_of_agents=15,
    distance_to_agents=0.4,
    distance_to_polygon=0.2,
    seed=1,
)
for position in positions:
    simulation.add_agent(
        jps.CollisionFreeSpeedModelAgentParameters(
            journey_id=journey_id,
            stage_id=wp1,
            position=position,
            radius=0.12,
        )
    )

while simulation.agent_count() > 0 and simulation.iteration_count() < 10_000:
    simulation.iterate()

print(f"Done in {simulation.iteration_count()} iterations.")

In [ ]:
from jupedsim.internal.notebook_utils import animate, read_sqlite_file

trajectory_data, walkable_area = read_sqlite_file(trajectory_file)
animate(trajectory_data, walkable_area)

## Example 2 – Least-targeted transition (load balancing)

Here a single distributing waypoint fans agents out to two downstream waypoints
(`wp_left` and `wp_right`) using `create_least_targeted_transition`.  Each
downstream waypoint then uses a fixed transition to a shared exit.  Because the
transition always picks the less-crowded branch, agents self-balance between the
two paths.


In [ ]:
trajectory_file2 = pathlib.Path("journey_lt.sqlite")

geometry2 = shapely.Polygon([(0, 0), (30, 0), (30, 10), (0, 10)])
simulation2 = jps.Simulation(
    model=jps.CollisionFreeSpeedModel(),
    geometry=geometry2,
    trajectory_writer=jps.SqliteTrajectoryWriter(
        output_file=trajectory_file2, commit_every_nth_write=1
    ),
)

# Distributing waypoint and two downstream branches
wp_dist = simulation2.add_waypoint_stage((10, 5), 1.0)
wp_left = simulation2.add_waypoint_stage((20, 8), 0.5)
wp_right = simulation2.add_waypoint_stage((20, 2), 0.5)
exit2 = simulation2.add_exit_stage([(29, 2), (30, 2), (30, 8), (29, 8)])

journey2 = jps.JourneyDescription([wp_dist, wp_left, wp_right, exit2])
journey2.set_transition_for_stage(
    wp_dist,
    jps.Transition.create_least_targeted_transition([wp_left, wp_right]),
)
journey2.set_transition_for_stage(
    wp_left, jps.Transition.create_fixed_transition(exit2)
)
journey2.set_transition_for_stage(
    wp_right, jps.Transition.create_fixed_transition(exit2)
)
journey2_id = simulation2.add_journey(journey2)

positions2 = jps.distributions.distribute_by_number(
    polygon=shapely.Polygon([(0.5, 0.5), (5, 0.5), (5, 9.5), (0.5, 9.5)]),
    number_of_agents=15,
    distance_to_agents=0.4,
    distance_to_polygon=0.2,
    seed=2,
)
for pos in positions2:
    simulation2.add_agent(
        jps.CollisionFreeSpeedModelAgentParameters(
            journey_id=journey2_id,
            stage_id=wp_dist,
            position=pos,
            radius=0.12,
        )
    )

while simulation2.agent_count() > 0 and simulation2.iteration_count() < 10_000:
    simulation2.iterate()

print(f"Done in {simulation2.iteration_count()} iterations.")

In [ ]:
from jupedsim.internal.notebook_utils import animate, read_sqlite_file

trajectory_data2, walkable_area2 = read_sqlite_file(trajectory_file2)
animate(trajectory_data2, walkable_area2)

## Try one change

In Example 1, replace the fixed transition on `wp1` with a round-robin
transition that cycles between `wp2` and `exit_id`:

```python
journey.set_transition_for_stage(
    wp1,
    jps.Transition.create_round_robin_transition([(wp2, 2), (exit_id, 1)]),
)
```

Two thirds of agents will still visit both waypoints; one third will skip `wp2`
and head straight to the exit.


## See also

- {doc}`Exit Stage </notebooks/fundamentals/03_exits>` – defining exit stages
- {doc}`Waypoint Stage </notebooks/fundamentals/04_waypoints>` – waypoint stages in detail
- {doc}`Routing </notebooks/fundamentals/08_routing>` – runtime re-routing with `switch_agent_journey`
- [JuPedSim scenario gallery](https://scenarios.jupedsim.org)
